# RockID · Stones · Dataset audit

Builds the training set for the 20 crystals and healing stones in `ml/configs/stones_v1.json` from three sources, then runs the same checks as the rock audit: duplicates, quality, background bias, leak-free train/val/test split.

| Source | Gives | Needs |
|---|---|---|
| Kaggle `lsind18/gemstones-images` | cut gems | `KAGGLE_USERNAME`, `KAGGLE_KEY` secrets |
| Roboflow Universe projects | crystals / gemstones | `ROBOFLOW_API_KEY` secret |
| Wikimedia Commons | licensed photos, mostly raw specimens | nothing |

**Before you run:** *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all*. If one source fails, the others still run; the error is shown in step 3.

**Outputs:** `MyDrive/RockID/stones_audit/<timestamp>/` (report; `review/` contact sheets stay on Drive) and `MyDrive/RockID/cache/stones_v1_raw.zip` (the images `02_train` uses). Step 10 downloads the files that belong in the repo.

## 1 · Settings

In [ ]:
REPO_URL = "https://github.com/GrowingRui/RockID.git"   # empty = use MyDrive/RockID/code/ml instead
REPO_BRANCH = "main"
DRIVE_ROOT = "/content/drive/MyDrive/RockID"
STONE_FILE = "configs/stones_v1.json"   # stones, aliases and sources (relative to ml/)
USE_KAGGLE, USE_ROBOFLOW, USE_COMMONS = True, True, True
RUN_EMBEDDINGS = True    # near-duplicates by visual similarity + probes (GPU)

## 2 · Setup: Drive, code, dependencies

In [ ]:
import importlib, json, os, pathlib, shutil, subprocess, sys, zipfile
from datetime import datetime, timezone
from google.colab import drive, userdata

drive.mount("/content/drive")
DRIVE = pathlib.Path(DRIVE_ROOT)
CACHE = DRIVE / "cache"; CACHE.mkdir(parents=True, exist_ok=True)

if REPO_URL:
    CODE = pathlib.Path("/content/rockid")
    url = REPO_URL
    try:
        url = REPO_URL.replace("https://", f"https://{userdata.get('GITHUB_TOKEN')}@")
    except Exception:
        pass  # public repo, no token needed
    cmd = ["git", "-C", str(CODE), "pull", "--ff-only"] if CODE.exists() else \
          ["git", "clone", "--depth", "1", "-b", REPO_BRANCH, url, str(CODE)]
    if subprocess.run(cmd, capture_output=True).returncode != 0:
        raise RuntimeError("git clone/pull failed. Check REPO_URL, REPO_BRANCH and the GITHUB_TOKEN secret.") from None
    ML = CODE / "ml"
else:
    ML = DRIVE / "code" / "ml"
assert (ML / "rockid_ml").exists(), f"rockid_ml not found under {ML}"
sys.path.insert(0, str(ML))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "imagehash>=4.3", "roboflow"], check=True)

import numpy as np, pandas as pd
import rockid_ml
from rockid_ml import audit, config, dedup, embed, images, probes, report, scan, sources, split
for m in (config, images, scan, dedup, embed, split, probes, sources, audit, report):
    importlib.reload(m)
from rockid_ml.config import AuditConfig

SC = json.loads((ML / STONE_FILE).read_text(encoding="utf-8"))
STONES = SC["stones"]
NAME = pathlib.Path(STONE_FILE).stem          # e.g. stones_v1
cfg = AuditConfig(label_from="folder", min_images_per_class=SC["min_images_per_class"])
print(f"{len(STONES)} stones · rockid_ml {rockid_ml.__version__} from {ML}")

## 3 · Download the sources (cached on Drive)

Each source is downloaded once and zipped to `MyDrive/RockID/cache/stones_src/`. Re-runs only unzip. Delete a source's `.zip` there to download it again.

In [ ]:
WORK = pathlib.Path("/content/data"); WORK.mkdir(parents=True, exist_ok=True)
SRC = WORK / "stones_src"; SRC.mkdir(exist_ok=True)
SRC_CACHE = CACHE / "stones_src"; SRC_CACHE.mkdir(exist_ok=True)

def cached_source(name, fetch):
    z, inv_csv, facts_json = SRC_CACHE / f"{name}.zip", SRC_CACHE / f"{name}.csv", SRC_CACHE / f"{name}.json"
    if z.exists() and inv_csv.exists():
        if not (SRC / name).exists():
            with zipfile.ZipFile(z) as f:
                f.extractall(SRC)
        return pd.read_csv(inv_csv), json.loads(facts_json.read_text()) if facts_json.exists() else {}
    inv, facts = fetch()
    tmp = WORK / f"{name}.zip"
    with zipfile.ZipFile(tmp, "w", zipfile.ZIP_STORED) as f:
        for p in inv["rel_path"]:
            f.write(SRC / p, p)
    shutil.copy(tmp, z)
    inv.to_csv(inv_csv, index=False)
    facts_json.write_text(json.dumps(facts, default=str))
    return inv, facts

inventories, source_facts, source_errors = [], {}, {}
def run_source(name, fetch):
    try:
        inv, facts = cached_source(name, fetch)
        inventories.append(inv); source_facts[name] = facts
        print(f"✓ {name}: {len(inv):,} images, {inv['source_class'].nunique()} classes")
    except Exception as e:
        source_errors[name] = f"{type(e).__name__}: {e}"[:500]
        print(f"✗ {name}: {source_errors[name]}")

def secret(name):
    """A Colab secret, or None if it doesn't exist or notebook access is off."""
    try:
        return userdata.get(name).strip()
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError) as e:
        print(f"Secret {name}: {type(e).__name__}")
        return None

def kaggle_credentials():
    """Colab secrets first; otherwise upload kaggle.json (as in the rock audit)."""
    user, key = secret("KAGGLE_USERNAME"), secret("KAGGLE_KEY")
    if user and key:
        return user, key
    print("Upload your kaggle.json now (kaggle.com → Settings → API → Create New Token).")
    from google.colab import files
    up = files.upload()
    if not up:
        raise RuntimeError("No kaggle.json uploaded.")
    creds = json.loads(next(iter(up.values())))
    return creds["username"], creds["key"]

if USE_KAGGLE:
    if not all((SRC_CACHE / f"kaggle-{ds.split('/')[-1]}.zip").exists() for ds in SC["sources"]["kaggle"]):  # cached: no login
        os.environ["KAGGLE_USERNAME"], os.environ["KAGGLE_KEY"] = kaggle_credentials()
    for ds in SC["sources"]["kaggle"]:
        run_source(f"kaggle-{ds.split('/')[-1]}", lambda ds=ds: (sources.fetch_kaggle(ds, SRC), {"dataset": ds}))
if USE_ROBOFLOW:
    rf_key = secret("ROBOFLOW_API_KEY")
    if not rf_key:
        from getpass import getpass
        rf_key = getpass("Paste your Roboflow Private API Key (hidden), then press Enter: ").strip()
    for proj in SC["sources"]["roboflow"]:
        run_source(f"roboflow-{proj.split('/')[1]}", lambda proj=proj: sources.fetch_roboflow(rf_key, proj, SRC))
if USE_COMMONS:
    run_source("commons", lambda: (sources.fetch_commons(STONES, SRC, SC["sources"]["commons_per_stone"],
                                                         SC["sources"]["commons_exclude_subcats"]), {"per_stone": SC["sources"]["commons_per_stone"]}))
assert inventories, "no source could be downloaded; see the errors above"
for name, f in source_facts.items():
    if name.startswith("roboflow"):
        print(name, "· type", f.get("type"), "· version", f.get("version"), "· license", f.get("license"))

## 4 · What the sources contain, and how it maps to our stones

Every class name found in every source, with the stone it was matched to. **Unmatched** classes are left out. If a stone you want is hiding under another name (e.g. `Tiger Iron`), add that name to the stone's `aliases` in `stones_v1.json` and re-run.

In [ ]:
inv = sources.label_inventory(pd.concat(inventories, ignore_index=True), STONES)
overview = sources.class_overview(inv)
matched = overview[overview["stone"].notna()]
unmatched = overview[overview["stone"].isna()]
print(f"{len(inv):,} images · {len(matched)} source classes matched to {matched['stone'].nunique()} stones · "
      f"{len(unmatched)} classes unmatched ({int(unmatched['images'].sum()):,} images)")
per_stone = inv[inv["label"].notna()].pivot_table(index="label", columns="source", values="rel_path", aggfunc="size", fill_value=0)
per_stone["total"] = per_stone.sum(1)
per_stone = per_stone.reindex([s["slug"] for s in STONES], fill_value=0).sort_values("total")
print("\nUnmatched classes (largest first):", ", ".join(f"{r.source_class} ({r.images})" for r in unmatched.sort_values("images", ascending=False).head(60).itertuples()))
per_stone

## 5 · Per-image stats: decode, size, blur, hashes, background (resumable)

In [ ]:
lab = inv[inv["label"].notna()].reset_index(drop=True)
df = pd.DataFrame({"rel_path": lab["rel_path"], "label": lab["label"], "label_raw": lab["source_class"],
                   "family": lab["source"], "file_bytes": [(SRC / p).stat().st_size for p in lab["rel_path"]]})
STATS_CSV = CACHE / f"{NAME}_image_stats.csv"
df = audit.analyze_all(SRC, df, cache_csv=STATS_CSV, plain_tol=cfg.plain_tol)
valid_mask = df["decode_error"].isna()
valid = df[valid_mask].reset_index(drop=True)
print(f"decoded {len(valid):,} / {len(df):,} · decode errors: {(~valid_mask).sum()}")

## 6 · Visual embeddings (GPU, cached)

In [ ]:
emb = emb_border = None
if RUN_EMBEDDINGS:
    import tensorflow as tf
    print("TF", tf.__version__, "· GPU:", tf.config.list_physical_devices("GPU") or "NONE, this will be slow")
    paths = [str(SRC / p) for p in valid["rel_path"]]
    emb = audit.cached_embeddings(CACHE / f"{NAME}_emb_full.npz", valid["rel_path"],
                                  lambda: embed.embed_paths(paths, size=cfg.embed_size))
    emb_border = audit.cached_embeddings(CACHE / f"{NAME}_emb_border.npz", valid["rel_path"],
                                         lambda: embed.embed_paths(paths, size=cfg.embed_size, variant="border"))
    print("embeddings:", emb.shape)

## 7 · Duplicates (also across sources), selection and split

Re-uploads are common (the same gem photo on Kaggle and Roboflow). A duplicate cluster always stays in one split, and clusters that carry two different stone names are excluded as label noise.

In [ ]:
pairs = audit.near_duplicate_pairs(valid, emb, cfg)
df["cluster_id"] = audit.assign_clusters(df, valid_mask, pairs)
sizes = df[valid_mask].groupby("cluster_id").size()
cross_source = df[valid_mask].groupby("cluster_id")["family"].nunique()
dup_stats = {
    "Exact-duplicate groups": int((valid.groupby("pixel_sha256").size() > 1).sum()),
    "Near-duplicate pairs (pHash)": int((pairs["kind"] == "phash").sum()),
    "Near-duplicate pairs (embedding)": int((pairs["kind"] == "embedding").sum()),
    "Clusters with more than one image": int((sizes > 1).sum()),
    "Clusters spanning two or more sources": int((cross_source > 1).sum()),
    "Largest cluster": int(sizes.max()),
}
d = audit.select_and_split(df, cfg)
kept = d[d["split"].notna()]
assert split.leaking_clusters(kept).empty, "a duplicate cluster spans several splits"
print(f"kept {len(kept):,} images in {kept['label_final'].nunique()} stones · leakage check passed")
print(d["excluded_reason"].value_counts().to_dict())
dup_stats

## 8 · Probes, class table, recommendations

In [ ]:
probe = bg_probe = nearest = confused = None
if emb is not None:
    pos = pd.Series(np.arange(len(valid)), index=valid["rel_path"])
    rows = pos[kept["rel_path"]].to_numpy()
    y, groups = kept["label_final"].to_numpy(), kept["cluster_id"].to_numpy()
    probe = probes.linear_probe(emb[rows], y, groups, seed=cfg.seed)
    nearest = probes.nearest_classes(emb[rows], y)
    confused = probes.confused_pairs(y, probe["pred"])
    bg_probe = probes.linear_probe(emb_border[rows], y, groups, seed=cfg.seed)
    print(f"probe top-1 {probe['top1']:.1%} · top-3 {report.pct(probe['top3'])} · chance {probe['chance']:.1%} · "
          f"background-only top-1 {bg_probe['top1']:.1%}")
t = audit.class_table(d, cfg, probe, nearest)
t = t.reindex([s["slug"] for s in STONES]).dropna(how="all")
recs = audit.recommendations(t, cfg, probe, bg_probe, int(d.loc[d["cross_class"], "cluster_id"].nunique()))
missing = [s["slug"] for s in STONES if s["slug"] not in t.index]
if missing:
    recs.insert(0, f"No images at all for: {', '.join(missing)}. Add aliases or another source.")
for r in recs:
    print("•", r)
t

## 9 · Report, contact sheets, training zip

In [ ]:
stamp = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M")
OUT = DRIVE / "stones_audit" / stamp
FIG, REV = OUT / "figures", OUT / "review"
FIG.mkdir(parents=True); REV.mkdir()
clean = d[d["excluded_reason"].isna()]

figures = {k: FIG / f"{k}.png" for k in ("class_counts", "plain_bg", "blur", "short_side")}
report.plot_class_counts(t, figures["class_counts"], cfg.min_images_per_class)
report.plot_share(t, "plain_bg_share", figures["plain_bg"], "share of plain-background images", cfg.studio_class_warn)
report.plot_hist(valid["blur_var"], figures["blur"], "Laplacian variance (higher = sharper)", cfg.blur_low, log=True)
report.plot_hist(np.minimum(valid["width"], valid["height"]), figures["short_side"], "short side (px)", cfg.min_side)

sheets = []
for stone, g in clean.groupby("label_final"):   # one sheet per stone, mixed sources: are these really that stone?
    sample = g.sample(min(32, len(g)), random_state=cfg.seed)
    sheets.append(report.contact_sheet([(str(SRC / p), src.split("-")[0]) for p, src in zip(sample["rel_path"], sample["family"])],
                                       REV / "stones" / f"{stone}.jpg"))
cross_ids = d.loc[d["cross_class"], "cluster_id"].unique()[:40]
sheets.append(report.contact_sheet(report.cluster_rows(d, SRC, cross_ids), REV / "cross_class.jpg", rows_of=6))
sheets = [s for s in sheets if s]

summary = {
    "Sources": "; ".join(f"{k}: {int((inv['source'] == k).sum()):,} images" for k in inv["source"].unique()),
    "Source errors": "; ".join(f"{k}: {v}" for k, v in source_errors.items()) or "none",
    "Matched to our stones": f"{int(inv['label'].notna().sum()):,} images from {len(matched)} source classes",
    "Clean images": f"{len(kept):,}",
    "Excluded": d["excluded_reason"].value_counts().to_dict(),
    "Stones below minimum": ", ".join(t.index[t["status"] == "below_min"]) or "none",
    "Stones with no images": ", ".join(missing) or "none",
    "Splits": kept["split"].value_counts().to_dict(),
    "Leakage check": "passed (no duplicate cluster spans two splits)",
}
raw_names = {s: sorted(g.unique().tolist()) for s, g in df.groupby("label")["label_raw"]}
path = report.write_report(OUT, {**cfg.to_dict(), "stone_file": STONE_FILE, "roboflow": source_facts}, summary, t,
                           None, [], raw_names, dup_stats, probe, bg_probe, confused, None, recs, sheets, figures)

manifest = audit.manifest(d)
manifest.to_csv(OUT / f"split_manifest_{NAME}.csv", index=False)
t.to_csv(OUT / "class_table.csv")
overview.to_csv(OUT / "source_classes.csv", index=False)
credits = inv.set_index("rel_path").loc[manifest["rel_path"], ["source", "license", "author", "url"]].reset_index()
credits.to_csv(OUT / f"{NAME}_sources.csv", index=False)

# The images 02_train needs, under the same relative paths as the manifest
RAW_ZIP = CACHE / f"{NAME}_raw.zip"
tmp = WORK / RAW_ZIP.name
with zipfile.ZipFile(tmp, "w", zipfile.ZIP_STORED) as z:
    for p in manifest["rel_path"]:
        z.write(SRC / p, p)
shutil.copy(tmp, RAW_ZIP)
print("Report:", path, "· training images:", RAW_ZIP)

## 10 · Download the files that belong in the repo

In [ ]:
from google.colab import files
bundle = WORK / f"rockid_stones_audit_{stamp}.zip"
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
    base_dir = f"reports/stones-audit-{stamp}"
    z.write(OUT / "audit.md", f"{base_dir}/audit.md")
    for f in FIG.glob("*.png"):
        z.write(f, f"{base_dir}/figures/{f.name}")
    for name in ("class_table.csv", "source_classes.csv"):
        z.write(OUT / name, f"{base_dir}/{name}")
    z.write(OUT / f"split_manifest_{NAME}.csv", f"splits/split_manifest_{NAME}.csv")
    z.write(OUT / f"{NAME}_sources.csv", f"splits/{NAME}_sources.csv")
files.download(str(bundle))
print("Send this zip to Claude, or unzip it into the repo's ml/ folder. review/ stays on Drive only.")

## Next
Look at `review/stones/<stone>.jpg` on Drive: do the photos really show that stone, and in the forms users will photograph (tumbled, cut)? Then share the zip. Training uses `02_train` with the stones config.